# 03 ― 地点予報を複数地点で比較

Open-Meteo Forecast API を呼んで、**3 地点 (徳島・札幌・那覇)** の気温・降水を並べて比較する。

ポイント:

- サービス層 (`open_meteo_forecast.fetch_forecast`) を直接 await する
- `asyncio.TaskGroup` で 3 地点を並列取得
- polars DataFrame に `location` 列を足して結合 → 一枚のプロットへ

**データ帰属**: 出典 Open-Meteo (https://open-meteo.com), CC-BY-4.0


## 3 地点を定義


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Spot:
    name: str
    latitude: float
    longitude: float

SPOTS = [
    Spot("徳島", 33.78, 134.49),
    Spot("札幌", 43.07, 141.35),
    Spot("那覇", 26.21, 127.68),
]


## 並列で 3 地点ぶん取得

Jupyter は top-level `await` をそのまま書ける ― `asyncio.run()` で包む必要は無い。


In [ ]:
import asyncio
import httpx
from aiseed_weather.services.open_meteo_forecast import fetch_forecast

async with httpx.AsyncClient() as client:
    async with asyncio.TaskGroup() as tg:
        tasks = {
            spot.name: tg.create_task(
                fetch_forecast(
                    latitude=spot.latitude,
                    longitude=spot.longitude,
                    client=client,
                    past_days=0,
                    forecast_days=7,
                )
            )
            for spot in SPOTS
        }

results = {name: task.result() for name, task in tasks.items()}
for name, r in results.items():
    print(f"  {name:6s}: model={r.model:12s}  rows={len(r.df):4d}  tz={r.timezone}")


## DataFrame を結合

各 `ForecastResult.df` は同じスキーマの polars DataFrame。`location` 列を追加して縦結合する。


In [ ]:
import polars as pl

df = pl.concat([
    r.df.with_columns(pl.lit(name).alias("location"))
    for name, r in results.items()
])
df.head(5)


## 気温 ― 3 地点の重ね描き


In [ ]:
import matplotlib.pyplot as plt

COLORS = {"徳島": "#234b86", "札幌": "#5a6fa5", "那覇": "#c0392b"}

fig, ax = plt.subplots(figsize=(11, 4.5))
for name in [s.name for s in SPOTS]:
    sub = df.filter(pl.col("location") == name)
    ax.plot(
        sub["timestamp"].to_list(),
        sub["temperature_2m"].to_list(),
        label=name, color=COLORS[name], linewidth=1.6,
    )
ax.set_ylabel("気温 / Temperature (°C)")
ax.set_title("7 日予報 ― 3 地点比較 (出典: Open-Meteo, CC-BY-4.0)")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()


## 日積算降水量 ― 棒グラフ比較


In [ ]:
# 各地点・各日の積算降水量 (mm/日)
daily = (
    df.with_columns(pl.col("timestamp").dt.date().alias("date"))
      .group_by(["location", "date"])
      .agg(pl.col("precipitation").sum().alias("daily_mm"))
      .sort(["date", "location"])
)

import numpy as np
dates = daily["date"].unique(maintain_order=True).to_list()
x = np.arange(len(dates))
width = 0.27

fig, ax = plt.subplots(figsize=(11, 4.5))
for i, name in enumerate([s.name for s in SPOTS]):
    vals = (
        daily.filter(pl.col("location") == name)
             .sort("date")["daily_mm"].to_list()
    )
    ax.bar(x + (i - 1) * width, vals, width,
           label=name, color=COLORS[name])
ax.set_xticks(x)
ax.set_xticklabels([d.strftime("%m/%d") for d in dates], rotation=30)
ax.set_ylabel("日降水量 / Daily precip (mm)")
ax.set_title("日積算降水量 ― 3 地点比較")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.show()


## 自分で試す

- `SPOTS` を編集して海外の地点を追加 (例: ロンドン `51.51, -0.13`)
- `forecast_days=15` まで延ばすと中期予報、`past_days=7` を入れると過去 1 週も取れる
- `model="jma_msm"` で気象庁 MSM、`model="gfs_seamless"` で GFS など切替可能
  (アクセス可能なモデル一覧は Open-Meteo ドキュメント参照)
- 気候値 (ERA5) と重ねたい場合は `point_climatology.hourly_climatology()` を使う
  ― 次のノート `04` 以降と組み合わせる
